In [81]:
!pip install torch torchvision tqdm

In [82]:
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

In [83]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

PyTorch: 2.13.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: NVIDIA A100 80GB PCIe


In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [85]:
data = torch.load('/home/jovyan/work/output_embeddings/2026-08-20T09:29:35.753419/dinov2_features.pt', map_location="cpu")
features = data['features']
paths = data['paths']
print("Features shape:", features.shape)
print("Number of paths:", len(paths))

Features shape: torch.Size([100863, 768])
Number of paths: 100863


In [86]:
class LstmModel(nn.Module):
    def __init__(self, feat_dim, lstm_hidden, num_layers, bidirectional, num_classes, dropout=0.0):
        super().__init__()

        self.lstm = nn.LSTM(
            feat_dim,
            lstm_hidden,
            num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden * (2 if bidirectional else 1), num_classes)

    def forward(self, x):
        seq, _ = self.lstm(x)
        return self.head(self.dropout(seq))

In [87]:
model = LstmModel(
    feat_dim=features.shape[-1],
    lstm_hidden=128,
    num_layers=1,
    bidirectional=True,
    num_classes=7,
    dropout=0.3,
)
model.to(device)

LstmModel(
  (lstm): LSTM(768, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (head): Linear(in_features=256, out_features=7, bias=True)
)

In [88]:
import os
import json
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, ConcatDataset


class CholecT50Feature:
    """CholecT50 loader that returns precomputed features instead of images.

    `features` is a tensor of shape [N, feat_dim] and `paths` is a parallel
    list of image paths (e.g. ".../VID01/000001.png") used to map each feature
    row back to its (video_id, frame_id).
    """

    def __init__(
        self,
        dataset_dir,
        features,
        paths,
        dataset_variant="cholect50-crossval",
        test_fold=1,
        sequence_length: int = 1,
    ):
        self.sequence_length = sequence_length
        if self.sequence_length < 1:
            raise ValueError("sequence_length must be >= 1")
        self.dataset_dir = dataset_dir
        self.features = features
        self.paths = paths

        # Build a global (video_id, frame_id) -> feature index lookup once.
        self.lookup = {}
        for idx, path in enumerate(paths):
            p = Path(path)
            self.lookup[(p.parent.name, p.stem)] = idx

        video_split = self._split_selector(case=dataset_variant)
        train_videos = (
            sum([v for k, v in video_split.items() if k != test_fold], [])
            if "crossval" in dataset_variant
            else video_split["train"]
        )
        test_videos = (
            sum([v for k, v in video_split.items() if k == test_fold], [])
            if "crossval" in dataset_variant
            else video_split["test"]
        )
        if "crossval" in dataset_variant:
            val_videos = train_videos[-5:]
            train_videos = train_videos[:-5]
        else:
            val_videos = video_split["val"]

        self.train_records = ["VID{}".format(str(v).zfill(2)) for v in train_videos]
        self.val_records = ["VID{}".format(str(v).zfill(2)) for v in val_videos]
        self.test_records = ["VID{}".format(str(v).zfill(2)) for v in test_videos]

        self.target_transform = self.to_binary
        self.build_train_dataset()
        self.build_val_dataset()
        self.build_test_dataset()

    def _split_selector(self, case="cholect50"):
        switcher = {
            "cholect50": {
                "train": [1, 15, 26, 40, 52, 65, 79, 2, 18, 27, 43, 56, 66, 92, 4, 22, 31, 47, 57, 68, 96, 5, 23, 35, 48, 60, 70, 103, 13, 25, 36, 49, 62, 75, 110],
                "val": [8, 12, 29, 50, 78],
                "test": [6, 51, 10, 73, 14, 74, 32, 80, 42, 111],
            },
            "cholect50-challenge": {
                "train": [1, 15, 26, 40, 52, 79, 2, 27, 43, 56, 66, 4, 22, 31, 47, 57, 68, 23, 35, 48, 60, 70, 13, 25, 49, 62, 75, 8, 12, 29, 50, 78, 6, 51, 10, 73, 14, 32, 80, 42],
                "val": [5, 18, 36, 65, 74],
                "test": [92, 96, 103, 110, 111],
            },
            "cholect45-crossval": {
                1: [79, 2, 51, 6, 25, 14, 66, 23, 50],
                2: [80, 32, 5, 15, 40, 47, 26, 48, 70],
                3: [31, 57, 36, 18, 52, 68, 10, 8, 73],
                4: [42, 29, 60, 27, 65, 75, 22, 49, 12],
                5: [78, 43, 62, 35, 74, 1, 56, 4, 13],
            },
            "cholect50-crossval": {
                1: [79, 2, 51, 6, 25, 14, 66, 23, 50, 111],
                2: [80, 32, 5, 15, 40, 47, 26, 48, 70, 96],
                3: [31, 57, 36, 18, 52, 68, 10, 8, 73, 103],
                4: [42, 29, 60, 27, 65, 75, 22, 49, 12, 110],
                5: [78, 43, 62, 35, 74, 1, 56, 4, 13, 92],
            },
            "cholect50-small": {
                "train": [1, 2],
                "val": [4],
                "test": [5],
            },
        }
        return switcher.get(case)

    def to_binary(self, label_list):
        outputs = []
        for label in label_list:
            label = torch.tensor(label).bool().int()
            outputs.append(label)
        return outputs

    def _make_video_dataset(self, video):
        label_file = os.path.join(self.dataset_dir, "labels", "{}.json".format(video))
        if self.sequence_length > 1:
            return T50SequenceFeature(
                video_id=video,
                features=self.features,
                lookup=self.lookup,
                label_file=label_file,
                window_size=self.sequence_length,
                target_transform=self.target_transform,
            )
        return T50Feature(
            video_id=video,
            features=self.features,
            lookup=self.lookup,
            label_file=label_file,
            target_transform=self.target_transform,
        )

    def build_train_dataset(self):
        iterable_dataset = []
        for video in self.train_records:
            dataset = self._make_video_dataset(video)
            if len(dataset) > 0:
                iterable_dataset.append(dataset)
        if len(iterable_dataset) == 0:
            raise ValueError("Train dataset is empty (check videos and sequence_length).")
        self.train_dataset = ConcatDataset(iterable_dataset)

    def build_val_dataset(self):
        iterable_dataset = []
        for video in self.val_records:
            dataset = self._make_video_dataset(video)
            if len(dataset) > 0:
                iterable_dataset.append(dataset)
        if len(iterable_dataset) == 0:
            raise ValueError("Val dataset is empty (check videos and sequence_length).")
        self.val_dataset = ConcatDataset(iterable_dataset)

    def build_test_dataset(self):
        iterable_dataset = []
        for video in self.test_records:
            dataset = self._make_video_dataset(video)
            if len(dataset) > 0:
                iterable_dataset.append(dataset)
        self.test_dataset = iterable_dataset

    def build(self):
        return (self.train_dataset, self.val_dataset, self.test_dataset)


class T50Feature(Dataset):
    """Single-frame feature dataset for one video."""

    def __init__(self, video_id, features, lookup, label_file, target_transform=None):
        label_data = json.load(open(label_file, "rb"))
        self.label_data = label_data["annotations"]
        self.frames = list(self.label_data.keys())
        self.video_id = video_id
        self.features = features
        self.lookup = lookup
        self.target_transform = target_transform

    def __len__(self):
        return len(self.frames)

    def get_binary_labels(self, labels):
        tool_label = np.zeros([6])
        verb_label = np.zeros([10])
        target_label = np.zeros([15])
        triplet_label = np.zeros([100])
        phase_label = np.zeros([7])
        for label in labels:
            triplet = label[0:1]
            if triplet[0] != -1.0:
                triplet_label[triplet[0]] += 1
            tool = label[1:7]
            if tool[0] != -1.0:
                tool_label[tool[0]] += 1
            verb = label[7:8]
            if verb[0] != -1.0:
                verb_label[verb[0]] += 1
            target = label[8:14]
            if target[0] != -1.0:
                target_label[target[0]] += 1
            phase = label[14:15]
            if phase[0] != -1.0:
                phase_label[phase[0]] += 1
        return (triplet_label, tool_label, verb_label, target_label, phase_label)

    def __getitem__(self, index):
        frame_key = self.frames[index]
        labels = self.label_data[frame_key]
        # Feature replaces the image.
        feature = self.features[self.lookup[(self.video_id, str(frame_key).zfill(6))]]
        if self.target_transform:
            labels = self.target_transform(self.get_binary_labels(labels))
        return feature, labels


class T50SequenceFeature(Dataset):
    """Contiguous feature windows from one video for temporal models (e.g. LSTM)."""

    def __init__(self, video_id, features, lookup, label_file, window_size, target_transform=None):
        if window_size < 2:
            raise ValueError("T50SequenceFeature requires window_size >= 2")
        label_data = json.load(open(label_file, "rb"))
        self.label_data = label_data["annotations"]
        self.frames = sorted(self.label_data.keys(), key=lambda k: int(k))
        self.video_id = video_id
        self.features = features
        self.lookup = lookup
        self.window_size = window_size
        self.target_transform = target_transform

    def __len__(self):
        return max(0, len(self.frames) - self.window_size + 1)

    def get_binary_labels(self, labels):
        tool_label = np.zeros([6])
        verb_label = np.zeros([10])
        target_label = np.zeros([15])
        triplet_label = np.zeros([100])
        phase_label = np.zeros([7])
        for label in labels:
            triplet = label[0:1]
            if triplet[0] != -1.0:
                triplet_label[triplet[0]] += 1
            tool = label[1:7]
            if tool[0] != -1.0:
                tool_label[tool[0]] += 1
            verb = label[7:8]
            if verb[0] != -1.0:
                verb_label[verb[0]] += 1
            target = label[8:14]
            if target[0] != -1.0:
                target_label[target[0]] += 1
            phase = label[14:15]
            if phase[0] != -1.0:
                phase_label[phase[0]] += 1
        return (triplet_label, tool_label, verb_label, target_label, phase_label)

    def __getitem__(self, index):
        frame_features = []
        stacked_components = [[] for _ in range(5)]
        for j in range(self.window_size):
            frame_key = self.frames[index + j]
            labels = self.label_data[frame_key]
            feature = self.features[self.lookup[(self.video_id, str(frame_key).zfill(6))]]
            frame_features.append(feature)
            label_tuple = self.get_binary_labels(labels)
            if self.target_transform:
                label_tuple = self.target_transform(label_tuple)
            for k, tensor in enumerate(label_tuple):
                stacked_components[k].append(tensor)
        features = torch.stack(frame_features, dim=0)  # [window, feat_dim]
        labels = tuple(torch.stack(parts, dim=0) for parts in stacked_components)
        return features, labels

In [89]:
# Instantiate the feature-based dataset with temporal windows so the LSTM
# sees real sequences (sequence_length > 1) instead of single frames.
# Adjust dataset_dir to the local CholecT50 root (contains labels/).
dataset_dir = "/home/jovyan/dataset/CholecT50"
SEQUENCE_LENGTH = 16

ds = CholecT50Feature(
    dataset_dir=dataset_dir,
    features=features,
    paths=paths,
    dataset_variant="cholect50-crossval",
    test_fold=1,
    sequence_length=SEQUENCE_LENGTH,
)

train_ds, val_ds, test_ds = ds.build()
print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

# Sanity check: a single sample returns (features, labels) with a sequence dim.
feat, labels = train_ds[0]
print("feature shape:", tuple(feat.shape))  # (L, feat_dim)
print("num label tensors:", len(labels))
print("phase label shape:", tuple(labels[4].shape))  # (L, 7)

train: 70243 val: 8125 test: 10
feature shape: (16, 768)
num label tensors: 5
phase label shape: (16, 7)


In [90]:
"""Metric computation for (optionally multi-head) CholecT50 classification.

Predictions and targets are collected per head as integer class indices over a
dataset and reduced at the end. Metric names follow ``ValidationConfig.metrics``:

- ``jaccard`` -> per-head mean per-class IoU (``<head>_jaccard``)
- ``f1_macro`` -> per-head macro F1 (``<head>_f1_macro``)
- ``f1_per_class`` -> per-head per-class F1 (``<head>_f1_class_<i>``)
- ``mean_accuracy`` -> per-head mean per-class accuracy (``<head>_mean_accuracy``)
- ``accuracy_per_class`` -> per-head per-class accuracy (``<head>_accuracy_class_<i>``)
- ``edit_score`` -> per-head sequence-level Levenshtein edit score (``<head>_edit_score``)
"""

from collections import defaultdict

import numpy as np


class MetricAccumulator:
    """Collect per-head prediction/target pairs and reduce to scalar metrics."""

    def __init__(self, heads: list[str], metric_names: list[str], num_classes: dict[str, int]):
        self.heads = list(heads)
        self.metric_names = list(metric_names)
        self.num_classes = num_classes
        self._pred: dict[str, list[np.ndarray]] = defaultdict(list)
        self._target: dict[str, list[np.ndarray]] = defaultdict(list)

    def _append(self, name: str, pred: np.ndarray, target: np.ndarray) -> None:
        self._pred[name].append(np.asarray(pred).reshape(-1))
        self._target[name].append(np.asarray(target).reshape(-1))

    def add_batch(
        self, logits: dict[str, np.ndarray], targets: dict[str, np.ndarray]
    ) -> None:
        for name in self.heads:
            if name not in logits or name not in targets:
                continue
            lo = np.asarray(logits[name])
            tg = np.asarray(targets[name])
            if lo.ndim == 3:  # (B, L, K) per-frame sequence -> (B*L, K)
                lo = lo.reshape(-1, lo.shape[-1])
                tg = tg.reshape(-1, tg.shape[-1])
            pred = np.argmax(lo, axis=1)
            tgt = np.argmax(tg, axis=1)
            self._append(name, pred, tgt)

    def compute(self) -> dict[str, float]:
        results: dict[str, float] = {}
        for name in self.heads:
            if name not in self._pred:
                continue
            pred = np.concatenate(self._pred[name]) if self._pred[name] else np.array([])
            target = (
                np.concatenate(self._target[name]) if self._target[name] else np.array([])
            )
            if pred.size == 0:
                continue
            n = int(self.num_classes.get(name, int(pred.max()) + 1))
            cm = _confusion_matrix(target, pred, n)
            for metric in self.metric_names:
                values = self._metric_values(metric, cm, pred, target, n)
                for key, value in values.items():
                    results[f"{name}_{key}"] = float(value)
        return results

    def _metric_values(
        self, metric: str, cm: np.ndarray, pred, target, n: int
    ) -> dict[str, float]:
        if metric == "edit_score":
            return {"edit_score": _edit_score(pred, target)}
        if metric == "jaccard":
            return {"jaccard": _jaccard_mean(cm)}
        if metric == "f1_macro":
            return {"f1_macro": _f1_macro(cm)}
        if metric == "mean_accuracy":
            return {"mean_accuracy": _mean_accuracy(cm)}
        if metric == "f1_per_class":
            f1s = _per_class_f1(cm)
            return {f"f1_class_{i}": float(v) for i, v in enumerate(f1s)}
        if metric == "accuracy_per_class":
            accs = _per_class_accuracy(cm)
            return {f"accuracy_class_{i}": float(v) for i, v in enumerate(accs)}
        raise ValueError(f"Unknown metric '{metric}'.")


def _confusion_matrix(target, pred, n: int) -> np.ndarray:
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (target, pred), 1)
    return cm


def _per_class_accuracy(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm)
    fn = cm.sum(axis=1) - tp
    return np.divide(
        tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0
    )


def _mean_accuracy(cm: np.ndarray) -> float:
    return float(_per_class_accuracy(cm).mean())


def _per_class_jaccard(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    union = tp + fp + fn
    return np.divide(tp, union, out=np.zeros_like(tp, dtype=float), where=union > 0)


def _jaccard_mean(cm: np.ndarray) -> float:
    return float(_per_class_jaccard(cm).mean())


def _per_class_f1(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    return np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )


def _f1_macro(cm: np.ndarray) -> float:
    return float(_per_class_f1(cm).mean())


def _edit_score(pred: np.ndarray, target: np.ndarray) -> float:
    """Sequence-level Edit Score = 1 - normalised Levenshtein distance.

    The predicted and target class sequences are compared with the Levenshtein
    distance (insertions + deletions + substitutions). The score lies in
    [0, 1] and is higher when the predicted sequence matches the ground-truth
    sequence more closely.
    """
    denom = max(len(pred), len(target))
    if denom == 0:
        return 0.0
    lev = _levenshtein(np.asarray(pred, dtype=np.int64), np.asarray(target, dtype=np.int64))
    return float(1.0 - lev / denom)


def _levenshtein(a: np.ndarray, b: np.ndarray) -> int:
    """Wagner-Fischer Levenshtein distance between two integer sequences."""
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    curr = [0] * (m + 1)
    for i in range(1, n + 1):
        ai = a[i - 1]
        curr[0] = i
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev, curr = curr, prev
    return prev[m]

In [91]:
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data.sampler import Sampler
from tqdm import tqdm

# --- Configuration ---------------------------------------------------------
# The dataset returns (features, labels) where features is (L, feat_dim) and
# labels is a tuple of 5 tensors each shaped (L, K). We train the phase head
# (index 4) per-frame over the sequence.
HEADS = ["phase"]
NUM_CLASSES = {"phase": 7}
METRICS = ["jaccard", "f1_macro", "mean_accuracy"]
# METRICS = ["jaccard", "f1_macro", "mean_accuracy", "edit_score"]
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-2


def collate_fn(batch):
    """Stack a batch of (features, labels) sequence samples.

    features is (L, feat_dim) per sample -> (B, L, feat_dim).
    labels is a tuple of 5 per-sample tensors shaped (L, K); stack each
    component across the batch -> list of 5 tensors shaped (B, L, K).
    """
    feats, labels = zip(*batch)
    feats = torch.stack(list(feats))  # (B, L, feat_dim)
    label_tensors = [torch.stack([l[i] for l in labels]) for i in range(5)]
    return feats, label_tensors


class VideoBatchSampler(Sampler):
    """Yield batches where every sample comes from the SAME video.

    The train/val/test splits are already per-video (each sub-dataset in the
    ConcatDataset is one video). This sampler groups indices by video so the
    LSTM never sees a batch that jumps between videos, preserving temporal
    coherence within each batch.
    """

    def __init__(self, concat_ds, batch_size, shuffle=True):
        self.batch_size = batch_size
        self.shuffle = shuffle
        # cumulative_sizes marks the end index of each sub-dataset (video).
        self.video_ranges = []
        start = 0
        for size in concat_ds.cumulative_sizes:
            self.video_ranges.append((start, size))
            start = size
        self.batches = self._build_batches()

    def _build_batches(self):
        batches = []
        for start, end in self.video_ranges:
            idxs = list(range(start, end))
            if self.shuffle:
                np.random.shuffle(idxs)
            for i in range(0, len(idxs), self.batch_size):
                batches.append(idxs[i:i + self.batch_size])
        if self.shuffle:
            np.random.shuffle(batches)  # shuffle batch order, not within-batch
        return batches

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)


# Wrap test (a list of per-video datasets) into a ConcatDataset for the sampler.
test_concat = ConcatDataset(test_ds)

train_loader = DataLoader(
    train_ds, batch_sampler=VideoBatchSampler(train_ds, BATCH_SIZE, shuffle=True),
    collate_fn=collate_fn, num_workers=0,
)
val_loader = DataLoader(
    val_ds, batch_sampler=VideoBatchSampler(val_ds, BATCH_SIZE, shuffle=False),
    collate_fn=collate_fn, num_workers=0,
)
test_loader = DataLoader(
    test_concat, batch_sampler=VideoBatchSampler(test_concat, BATCH_SIZE, shuffle=False),
    collate_fn=collate_fn, num_workers=0,
)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# --- Class-balanced cross-entropy ------------------------------------------
# Compute inverse class-frequency weights from the training split so rare
# phases aren't neglected (this directly boosts macro F1). We iterate the
# raw per-video label dicts (no feature loading) to count phases quickly.
def _phase_counts(concat_ds):
    counts = torch.zeros(NUM_CLASSES["phase"])
    for sub in concat_ds.datasets:              # each is a T50SequenceFeature
        label_data = sub.label_data
        frames = sub.frames
        for fk in frames:
            for label in label_data[fk]:
                p = label[14:15]
                if p[0] != -1.0:
                    counts[int(p[0])] += 1
    return counts

phase_counts = _phase_counts(train_ds)
phase_freq = (phase_counts / phase_counts.sum()).clamp(min=1e-9)
class_weights = 1.0 / phase_freq
# Normalise so mean weight ~ 1.
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)
print("Per-class phase counts:", phase_counts.int().tolist())
print("Phase frequencies:     ", [f"{f:.3f}" for f in phase_freq.tolist()])
print("Used class weights:    ", [f"{w:.2f}" for w in class_weights.cpu().tolist()])

criterion = nn.CrossEntropyLoss(weight=class_weights)


def run_epoch(loader, train=True, desc=""):
    model.train() if train else model.eval()
    accumulator = MetricAccumulator(HEADS, METRICS, NUM_CLASSES)
    total_loss = 0.0
    n_samples = 0

    with torch.set_grad_enabled(train):
        pbar = tqdm(loader, desc=desc, leave=False)
        for feats, label_tensors in pbar:
            feats = feats.to(device)          # (B, L, feat_dim)
            phase = label_tensors[4].to(device)  # (B, L, 7) one-hot

            # LSTM expects (B, L, feat); returns per-frame logits (B, L, 7).
            logits = model(feats)             # (B, L, 7)

            # Per-frame loss over the whole sequence (class-weighted).
            loss = criterion(
                logits.reshape(-1, NUM_CLASSES["phase"]),
                phase.argmax(dim=-1).reshape(-1),
            )

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * feats.size(0)
            n_samples += feats.size(0)

            accumulator.add_batch(
                {"phase": logits.detach().cpu().numpy()},
                {"phase": phase.detach().cpu().numpy()},
            )

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    metrics = accumulator.compute()
    return total_loss / n_samples, metrics


for epoch in range(1, EPOCHS + 1):
    train_loss, train_metrics = run_epoch(train_loader, train=True, desc=f"train {epoch}/{EPOCHS}")
    val_loss, val_metrics = run_epoch(val_loader, train=False, desc=f"val   {epoch}/{EPOCHS}")
    print(f"Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")
    print(f"  train: {train_metrics}")
    print(f"  val:   {val_metrics}")

# Final evaluation on the held-out test split.
test_loss, test_metrics = run_epoch(test_loader, train=False, desc="test")
print(f"\nTest loss={test_loss:.4f}")
print(f"Test metrics: {test_metrics}")

Per-class phase counts: [4326, 51422, 9361, 30505, 4759, 8176, 4432]
Phase frequencies:      ['0.038', '0.455', '0.083', '0.270', '0.042', '0.072', '0.039']
Used class weights:     ['1.71', '0.14', '0.79', '0.24', '1.55', '0.90', '1.67']


Epoch 1/10  train_loss=0.9809  val_loss=1.1946
  train: {'phase_jaccard': 0.391585915864186, 'phase_f1_macro': 0.5594640801049187, 'phase_mean_accuracy': 0.6414244121865417}
  val:   {'phase_jaccard': 0.37626068998613454, 'phase_f1_macro': 0.5404753462476058, 'phase_mean_accuracy': 0.6144555899172263}


Epoch 2/10  train_loss=0.8183  val_loss=1.1394
  train: {'phase_jaccard': 0.466652795523159, 'phase_f1_macro': 0.6333832790185332, 'phase_mean_accuracy': 0.7100187100367019}
  val:   {'phase_jaccard': 0.39637865006588396, 'phase_f1_macro': 0.5611806634831155, 'phase_mean_accuracy': 0.6322875736780915}


Epoch 3/10  train_loss=0.7512  val_loss=1.1073
  train: {'phase_jaccard': 0.49729184473386734, 'phase_f1_macro': 0.6613971433032741, 'phase_mean_accuracy': 0.7359401684958496}
  val:   {'phase_jaccard': 0.41042111484210914, 'phase_f1_macro': 0.5758949146410377, 'phase_mean_accuracy': 0.640399485113021}


Epoch 4/10  train_loss=0.7485  val_loss=1.0763
  train: {'phase_jaccard': 0.4986692651776018, 'phase_f1_macro': 0.6626372051348132, 'phase_mean_accuracy': 0.7391953583060801}
  val:   {'phase_jaccard': 0.42127894214246525, 'phase_f1_macro': 0.5869851579325692, 'phase_mean_accuracy': 0.6420546368866944}


Epoch 5/10  train_loss=0.7263  val_loss=1.0932
  train: {'phase_jaccard': 0.5074808865800157, 'phase_f1_macro': 0.6706251636035804, 'phase_mean_accuracy': 0.745904709830479}
  val:   {'phase_jaccard': 0.41298226544447175, 'phase_f1_macro': 0.5775635496313389, 'phase_mean_accuracy': 0.6407021749007787}


Epoch 6/10  train_loss=0.7181  val_loss=1.0674
  train: {'phase_jaccard': 0.511813486335179, 'phase_f1_macro': 0.6744062976876507, 'phase_mean_accuracy': 0.7489717268864506}
  val:   {'phase_jaccard': 0.4040690072306799, 'phase_f1_macro': 0.5696500418142222, 'phase_mean_accuracy': 0.6408441170683199}


Epoch 7/10  train_loss=0.6848  val_loss=1.0719
  train: {'phase_jaccard': 0.5270496852912836, 'phase_f1_macro': 0.6879234786323183, 'phase_mean_accuracy': 0.7617657016104741}
  val:   {'phase_jaccard': 0.4100227123945851, 'phase_f1_macro': 0.5744149988290814, 'phase_mean_accuracy': 0.641751523624384}


Epoch 9/10  train_loss=0.6574  val_loss=1.0045
  train: {'phase_jaccard': 0.543419737748632, 'phase_f1_macro': 0.7020735319626483, 'phase_mean_accuracy': 0.7746893763604212}
  val:   {'phase_jaccard': 0.4381141539210943, 'phase_f1_macro': 0.601202783087423, 'phase_mean_accuracy': 0.6451965842428241}


train 10/10:  17%|█▋        | 366/2209 [00:14<01:10, 26.12it/s, loss=0.5746]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

